# Memory Governance — Dev Log

## Objetivo

`MemoryStore.store()` escaneia todo conteúdo por PII real
(`pii_detection.detect`) antes de persistir, e redige qualquer achado — o
texto original com PII nunca toca o disco. Comportamento padrão obrigatório,
não configurável.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.memory_governance.store import MemoryStore

demo_dir = Path(tempfile.mkdtemp(prefix="memory_demo_"))
store = MemoryStore(storage_path=demo_dir / "memory.json")

item1 = store.store("Usuário perguntou sobre horário de atendimento.")
item2 = store.store("Cliente João Silva, CPF 111.444.777-35, relatou problema no pedido #4521.", ttl_days=30)

print(f"Item 1 (sem PII): redacted={item1.redacted} | conteúdo: {item1.content}")
print(f"Item 2 (com CPF): redacted={item2.redacted} | categoria={item2.category.value}")
print(f"  conteúdo armazenado: {item2.content}")
print(f"  expira em: {item2.expires_at}")

Item 1 (sem PII): redacted=False | conteúdo: Usuário perguntou sobre horário de atendimento.
Item 2 (com CPF): redacted=True | categoria=personal
  conteúdo armazenado: Cliente João Silva, CPF [REDACTED:CPF], relatou problema no pedido #4521.
  expira em: 2026-09-20 00:04:12.501934+00:00


O CPF real foi removido do texto persistido (`[REDACTED:CPF]`), mas o
resto do conteúdo — nome, número do pedido — foi preservado, exatamente o
comportamento desejado: redação cirúrgica, não descarte do item inteiro.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/memory_governance/tests -v
```

8/8 testes passando.